In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('telecom_ibm_eda_completed.csv')
print(df.shape)
df.head()

(7043, 25)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_Binary,Name,Age,SupportTicketCount
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,Şilan Bilge,56,5
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,One year,No,Mailed check,56.95,1889.50,No,0,Taygan Gül,69,3
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,Fami Sezer,46,8
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,Denizalp Yıldırım,32,2
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,Günsel Akar,60,4


In [2]:
print("Doldurmadan önce eksik:", df['TotalCharges'].isnull().sum())

df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("Doldurduktan sonra eksik:", df['TotalCharges'].isnull().sum())

Doldurmadan önce eksik: 11
Doldurduktan sonra eksik: 0


In [3]:
Q1 = df['SupportTicketCount'].quantile(0.25)
Q3 = df['SupportTicketCount'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

print("Üst sınır:", upper_limit)
print("Capping öncesi max:", df['SupportTicketCount'].max())

df['SupportTicketCount'] = df['SupportTicketCount'].clip(upper=upper_limit)

print("Capping sonrası max:", df['SupportTicketCount'].max())

Üst sınır: 6.0
Capping öncesi max: 11
Capping sonrası max: 6


In [4]:
# İkili (Yes/No) kolonlar
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col + '_Binary'] = df[col].map({'Yes': 1, 'No': 0})

df['Gender_Binary'] = df['gender'].map({'Male': 1, 'Female': 0})

df[binary_cols + ['gender']].head()

,Partner,Dependents,PhoneService,PaperlessBilling,Churn,gender
0,Yes,No,No,Yes,No,Female
1,No,No,Yes,No,No,Male
2,No,No,Yes,Yes,Yes,Male
3,No,No,No,No,No,Male
4,No,No,Yes,Yes,Yes,Female


In [5]:
# Çok kategorili kolonlar -> One-Hot Encoding
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
                   'OnlineBackup', 'DeviceProtection', 'TechSupport',
                   'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

df_encoded = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

print("Öncesi:", df.shape)
print("Sonrası:", df_encoded.shape)

Öncesi: (7043, 30)
Sonrası: (7043, 41)


In [6]:
def tenure_group(months):
    if months <= 12:
        return '0-1 Yıl'
    elif months <= 24:
        return '1-2 Yıl'
    elif months <= 48:
        return '2-4 Yıl'
    else:
        return '4+ Yıl'

df_encoded['TenureGroup'] = df_encoded['tenure'].apply(tenure_group)
df_encoded['TenureGroup'].value_counts()

TenureGroup
4+ Yıl     2239
0-1 Yıl    2186
2-4 Yıl    1594
1-2 Yıl    1024
Name: count, dtype: int64

In [7]:
def customer_segment(row):
    if row['MonthlyCharges'] > 70 and row['tenure'] > 24:
        return 'Yüksek Değerli Sadık Müşteri'
    elif row['MonthlyCharges'] > 70 and row['tenure'] <= 24:
        return 'Yüksek Değerli Yeni Müşteri'
    elif row['MonthlyCharges'] <= 70 and row['tenure'] > 24:
        return 'Düşük Değerli Sadık Müşteri'
    else:
        return 'Düşük Değerli Yeni Müşteri'

df_encoded['CustomerSegment'] = df_encoded.apply(customer_segment, axis=1)
df_encoded['CustomerSegment'].value_counts()

CustomerSegment
Yüksek Değerli Sadık Müşteri    2230
Düşük Değerli Yeni Müşteri      1857
Düşük Değerli Sadık Müşteri     1603
Yüksek Değerli Yeni Müşteri     1353
Name: count, dtype: int64

In [9]:
print(df_encoded.shape)
df_encoded.to_csv('telecom_ibm_feature_engineered.csv', index=False)   #kaydetme komutu çalıştırmadım

(7043, 43)
